In [1]:
import sys
print(sys.executable)

c:\Users\chine\enterprise-policy-assistant\venv\Scripts\python.exe


In [2]:
import langchain
print(langchain.__version__)

1.2.17


In [3]:
import os
from dotenv import load_dotenv

# Document Loading & Splitting
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings & Vector Store
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

# Retrieval Chain Components(LCEL)
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

True

In [4]:
# Load PDF
loader = PyPDFLoader("../data/raw/employee_handbook.pdf")
documents = loader.load()

# Split documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

In [5]:
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print(api_key[:7] + "...")
else:
    print("API key not found.")

sk-proj...


In [6]:
# Create vector store
embeddings = OpenAIEmbeddings()

# Create vector store
vectorstore = FAISS.from_documents(chunks,embeddings)
vectorstore.save_local('../data/processed/faiss_index')




In [7]:
# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


In [8]:
#1. Define question(prompt)
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)


#2. Setup the retriever
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

# 3. Create the chain (LCEL)
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Run the query
query = 'What is the company remote work policy?'
response = chain.invoke(query)
print(response)

The provided context does not mention a remote work policy for the company. It primarily discusses the work environment, project selection, and the flexibility of moving desks within the office.
